In [ ]:
import io
import boto3
import pandas as pd
from sqlalchemy import create_engine

Conectar com DataLake usando boto3 e lendo parquet

In [ ]:
# Configurações do DataLake - SupabaseS3
S3_ENDPOINT_URL = "https://XXXX.storage.supabase.co/storage/v1/s3"
AWS_REGION = "us-east-1"
AWS_ACCESS_KEY_ID = "XXXX"
AWS_SECRET_ACCESS_KEY = "XXXXX"
BUCKET_NAME = "XXXX"

In [ ]:
s3 = boto3.client(
    "s3",
    region_name=AWS_REGION,
    endpoint_url=S3_ENDPOINT_URL,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

# print(type(s3))

In [ ]:
# Listar arquivos no bucket
response = s3.list_objects(Bucket=BUCKET_NAME)
arquivos = [obj["Key"] for obj in response["Contents"]]

In [ ]:
print(arquivos)

___

Ler as tabelas do datalake

In [ ]:
TABELAS = [
    'produtos',
    'clientes',
    'vendas',
    'preco_competidores',
]

In [ ]:
dataframes = {}

for tabela in TABELAS:
    print(f'Baixando {tabela}.parquet do DataLake...')

    file_key = f"{tabela}.parquet"

    # Baixar o arquivo do S3
    response = s3.get_object(Bucket=BUCKET_NAME, Key=file_key)
    parquet_bytes = response["Body"].read()

    # Converter bytes → DataFrame e guardar no dicionário
    dataframes[tabela] = pd.read_parquet(io.BytesIO(parquet_bytes))

    print(f"✅ {tabela}: {len(dataframes[tabela])} linhas carregadas")

___

Salvar no PostgreSQL (Supabase)

In [ ]:
DATABASE_URL = 'postgresql://xxxx'

In [ ]:
engine = create_engine(DATABASE_URL)

In [ ]:
for tabela, df in dataframes.items():
    print(f"💾 Salvando {tabela} no PostgreSQL...")

    df.to_sql(
        tabela,  # Nome da tabela no banco
        engine,  # Engine de conexão
        if_exists="replace",  # Substituir se existir
        index=False  # Não salvar índice do pandas
    )

    print(f"✅ {tabela}: {len(df)} linhas salvas no banco")

In [ ]:
# Verificando se os dados foram salvos corretamente

print("\n📊 Verificação final:")
for tabela in TABELAS:
    df_verificacao = pd.read_sql_query(f"SELECT COUNT(*) as total FROM {tabela}", engine)
    total = df_verificacao["total"].iloc[0]
    print(f"  ✅ {tabela}: {total} linhas no banco")


In [ ]:
# Fechar conexão
engine.dispose()